In [28]:
import pandas as pd
import scipy as sp
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.metrics import r2_score, mean_squared_error

import astropy.constants as const

import joblib

In [29]:
Erg_to_J = 10**-7    # J per erg
cm2_to_m2 = 1/(100**2)  # m2 per cm2
parsec_to_m = 30856775814913673   # m per parsec
SolarRadii_to_m = 6.957*(10**8) # m per solar radii
AU_to_m = 149597870700 # m per AU

In [30]:
def xi(gamma, tau):
    return (2.0/3) * (1 + (1/gamma) * (1 + (0.5*gamma*tau-1)*np.exp(-gamma*tau)) +
                        gamma*(1 - 0.5*tau**2) * sp.special.expn(2, gamma*tau)             )    

def PT_line(pressure, params, R_star, T_star, T_int, sma, grav):
    kappa  = 10**(params[0])
    gamma1 = 10**(params[1])
    gamma2 = 10**(params[2])
    alpha, beta = params[3], params[4]

    # Stellar input temperature (at top of atmosphere):
    T_irr = beta * (R_star / (2.0*sma))**0.5 * T_star

    # Gray IR optical depth:
    tau = kappa * (pressure*1e6) / grav # Convert bars to barye (CGS)

    xi1 = xi(gamma1, tau)
    xi2 = xi(gamma2, tau)

    # Temperature profile (Eq. 13 of Line et al. 2013):
    temperature = (0.75 * (T_int**4 * (2.0/3.0 + tau) +
                             T_irr**4 * (1-alpha) * xi1 +
                             T_irr**4 * alpha     * xi2 ) )**0.25

    return temperature

In [31]:
r_Sun = SolarRadii_to_m
A_Sun = 4*np.pi*(r_Sun**2)

T_Sun = 5772 # Effective BB temperature in K
SB_const =  5.670374419*(10**-8) 
S_Sun = SB_const*(T_Sun**4)

# Data
Planets = ['Mercury', 'Venus', 'Earth', 'Mars']
SMA = [57909175678.25, 108208925513.19, 149597887155.77, 227936637241.84]  # mm

T = [167 + 273.15, 464 + 273.15, 15 + 273.15, -63 + 273.15]  # Kelvin
P = [0.000000000000005, 92, 1.01325, 0.00636]  # bar
albedo_TOA  = [0.12 ,0.75, 0.31, 0.25] # ETH Zurich
# albedo_surf = [0.06, 0, 0, 0.29]

# Gas mole fractions
# Order: CO2, N2, O2, Ar, CH4, Na, H2, He, Other
x_Mercury = [0, 0, 0.42, 0, 0, 0.22, 0.22, 0.06, 0.08]
x_Venus   = [0.96, 0.04, 0, 0, 0, 0, 0, 0, 0]
x_Earth   = [420/1e6, 0.78, 0.21, 0.01, 0, 0, 0, 0, 0.01]
x_Mars    = [0.95, 0.027, 0, 0.016, 0, 0, 0, 0, 0.007]

gas_names = ["CO2", "N2", "O2", "Ar", "CH4", "Na", "H2", "He", "Other"]

gas_data = [
    x_Mercury,
    x_Venus,
    x_Earth,
    x_Mars
]

# Build DataFrame
df = pd.DataFrame({
    "SMA (m)": SMA,
    "T_actual (K)": T,
    "planet_surface_pressure_bars": P,
    "albedo_TOA": albedo_TOA
}, index = Planets)

# Add gas fraction columns
for i, gas in enumerate(gas_names):
    df[gas] = [planet_gases[i] for planet_gases in gas_data]

df['pH2O (bar)'] = 1e-12
df['pCO2 (bar)'] = df['planet_surface_pressure_bars']*df['CO2']
df['pO2 (bar)'] = df['planet_surface_pressure_bars']*df['O2']
df['pN2 (bar)'] = df['planet_surface_pressure_bars']*df['N2']
df['pCH4 (bar)'] = 1e-12
df['pN2O (bar)'] = 1e-12
df['pCO (bar)'] = 1e-12
df['pO3 (bar)'] = 1e-12
df['pSO2 (bar)'] = 1e-12
df['pNH3 (bar)'] = 1e-12
df['pC2H6 (bar)'] = 1e-12
df['pNO2 (bar)'] = 1e-12

df

,SMA (m),T_actual (K),planet_surface_pressure_bars,albedo_TOA,CO2,N2,O2,Ar,CH4,Na,...,pO2 (bar),pN2 (bar),pCH4 (bar),pN2O (bar),pCO (bar),pO3 (bar),pSO2 (bar),pNH3 (bar),pC2H6 (bar),pNO2 (bar)
Mercury,5.790918e+10,440.15,5.000000e-15,0.12,0.00000,0.000,0.42,0.000,0,0.22,...,2.100000e-15,0.000000,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12
Venus,1.082089e+11,737.15,9.200000e+01,0.75,0.96000,0.040,0.00,0.000,0,0.00,...,0.000000e+00,3.680000,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12
Earth,1.495979e+11,288.15,1.013250e+00,0.31,0.00042,0.780,0.21,0.010,0,0.00,...,2.127825e-01,0.790335,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12
Mars,2.279366e+11,210.15,6.360000e-03,0.25,0.95000,0.027,0.00,0.016,0,0.00,...,0.000000e+00,0.000172,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12,1.000000e-12


In [32]:
# def BB_EB(d_SMA, alpha):
#     T = ((1/4) * (S_Sun*A_Sun) / (SB_const*4*np.pi*(d_SMA**2)) * (1 - alpha))**0.25
#     return T


# def F_EB(d_SMA, alpha):
#     LR = joblib.load('../Approximations for OLR/T/BestModel1.pkl')

#     F = (1/4) * ((S_Sun*A_Sun)/(4*np.pi*d_SMA**2)) * (1 - alpha)

#     X = pd.DataFrame({
#         "True Planet Signal (W/m2)": F
#     })

#     T_log = LR.predict(X)
#     return 10**T_log


# def F_TBB_EB(d_SMA, alpha):
#     LR = joblib.load('../Approximations for OLR/T/BestModel2.pkl')

#     F = (1/4) * ((S_Sun*A_Sun)/(4*np.pi*d_SMA**2)) * (1 - alpha)
#     T_eff_BB = (F / SB_const)**0.25

#     X = pd.DataFrame({
#         "True Planet Signal (W/m2)": F,
#         "T_eff BB (K)": T_eff_BB
#     })

#     T_log = LR.predict(X)
#     return 10**T_log


# def F_TBB_p_EB(d_SMA, alpha, p):
#     LR = joblib.load('../Approximations for OLR/T/BestModel14.pkl')

#     F = (1/4) * ((S_Sun*A_Sun)/(4*np.pi*d_SMA**2)) * (1 - alpha)
#     T_eff_BB = (F / SB_const)**0.25

#     X = pd.DataFrame({
#         "True Planet Signal (W/m2)": F,
#         "T_eff BB (K)": T_eff_BB,
#         "pH2O (bar)": p[:,0],
#         "pCO2 (bar)": p[:,1],
#         "pO2 (bar)": p[:,2],
#         "pN2 (bar)": p[:,3],
#         "pCH4 (bar)": p[:,4],
#         "pN2O (bar)": p[:,5],
#         "pCO (bar)": p[:,6],
#         "pO3 (bar)": p[:,7],
#         "pSO2 (bar)": p[:,8],
#         "pNH3 (bar)": p[:,9],
#         "pC2H6 (bar)": p[:,10],
#         "pNO2 (bar)": p[:,11]
#     })

#     T_log = LR.predict(X)
#     return 10**T_log


# def F_TGB_p_EB(d_SMA, alpha, p):
#     LR = joblib.load('../Approximations for OLR/T/BestModel15.pkl')

#     E = 1 - alpha  # TOA emissivity

#     F = (1/4) * ((S_Sun*A_Sun)/(4*np.pi*d_SMA**2)) * (1 - alpha)
#     T_eff_GB = (F / (E * SB_const))**0.25

#     X = pd.DataFrame({
#         "True Planet Signal (W/m2)": F,
#         "pH2O (bar)": p[:,0],
#         "pCO2 (bar)": p[:,1],
#         "pO2 (bar)": p[:,2],
#         "pN2 (bar)": p[:,3],
#         "pCH4 (bar)": p[:,4],
#         "pN2O (bar)": p[:,5],
#         "pCO (bar)": p[:,6],
#         "pO3 (bar)": p[:,7],
#         "pSO2 (bar)": p[:,8],
#         "pNH3 (bar)": p[:,9],
#         "pC2H6 (bar)": p[:,10],
#         "pNO2 (bar)": p[:,11],
#         "T_eff GB (K)": T_eff_GB
#     })

#     T_log = LR.predict(X)
#     return 10**T_log


In [33]:
# df['BB'] = BB_EB(df['SMA (m)'], df['albedo_TOA'])
# df['C1'] = F_EB(df['SMA (m)'], df['albedo_TOA'])
# df['C2'] = F_TBB_EB(df['SMA (m)'], df['albedo_TOA'])
# df['C3'] = F_TBB_p_EB(df['SMA (m)'], df['albedo_TOA'], [df['pH2O (bar)'], df['pCO2 (bar)'], df['pO2 (bar)'], df['pN2 (bar)'], df['pCH4 (bar)'], df['pN2O (bar)'], df['pCO (bar)'], df['pO3 (bar)'], df['pSO2 (bar)'], df['pNH3 (bar)'], df['pC2H6 (bar)'], df['pNO2 (bar)']])
# df['C4'] = F_TGB_p_EB(df['SMA (m)'], df['albedo_TOA'], [df['pH2O (bar)'], df['pCO2 (bar)'], df['pO2 (bar)'], df['pN2 (bar)'], df['pCH4 (bar)'], df['pN2O (bar)'], df['pCO (bar)'], df['pO3 (bar)'], df['pSO2 (bar)'], df['pNH3 (bar)'], df['pC2H6 (bar)'], df['pNO2 (bar)']])

LR1 = joblib.load('../Approximations for OLR/T/Complexity_1_Model.pkl')
LR2 = joblib.load('../Approximations for OLR/T/Complexity_2_Model.pkl')
LR3 = joblib.load('../Approximations for OLR/T/Complexity_3_Model.pkl')
LR4 = joblib.load('../Approximations for OLR/T/Complexity_4_Model.pkl')

df['BB'] = ((1/4) * (S_Sun*A_Sun) / (SB_const*4*np.pi*(df['SMA (m)']**2)) * (1 - df['albedo_TOA']))**0.25
F = (1/4) * ((S_Sun*A_Sun)/(4*np.pi*(df['SMA (m)']**2))) * (1 - df['albedo_TOA'])

X = pd.DataFrame({
    "True Planet Signal (W/m2)^-3": np.power(F,-3),
    "True Planet Signal (W/m2)^-2": np.power(F,-2),
    "True Planet Signal (W/m2)^-1": np.power(F,-1),
    "True Planet Signal (W/m2)^1": np.power(F,1),
    "True Planet Signal (W/m2)^2": np.power(F,2),
    "True Planet Signal (W/m2)^3": np.power(F,3)
})
T = LR1.predict(X)
df['C1'] = T

X = pd.DataFrame({
    "True Planet Signal (W/m2)^-3": np.power(F,-3),
    "True Planet Signal (W/m2)^-2": np.power(F,-2),
    "True Planet Signal (W/m2)^-1": np.power(F,-1),
    "True Planet Signal (W/m2)^1": np.power(F,1),
    "True Planet Signal (W/m2)^2": np.power(F,2),
    "True Planet Signal (W/m2)^3": np.power(F,3)


    #"T_eff BB (K)": (F / SB_const)**0.25
})
T = LR2.predict(X)
df['C2'] = T

X = pd.DataFrame({
    "True Planet Signal (W/m2)": F,
    "T_eff BB (K)": (F / SB_const)**0.25,
    "pH2O (bar)": df['pH2O (bar)'],
    "pCO2 (bar)": df['pCO (bar)'],
    "pO2 (bar)": df['pO2 (bar)'],
    "pN2 (bar)": df['pN2 (bar)'],
    "pCH4 (bar)": df['pCH4 (bar)'],
    "pN2O (bar)": df['pN2O (bar)'],
    "pCO (bar)": df['pCO (bar)'],
    "pO3 (bar)": df['pO3 (bar)'],
    "pSO2 (bar)": df['pSO2 (bar)'],
    "pNH3 (bar)": df['pNH3 (bar)'],
    "pC2H6 (bar)": df['pC2H6 (bar)'],
    "pNO2 (bar)": df['pNO2 (bar)']
})
X = X.replace(0, 1e-12)
T_log = LR3.predict(X)
df['C3'] = np.power(10, T_log)

E = 1 - df['albedo_TOA']  # TOA emissivity
X = pd.DataFrame({
    "True Planet Signal (W/m2)": F,
    "pH2O (bar)": df['pH2O (bar)'],
    "pCO2 (bar)": df['pCO (bar)'],
    "pO2 (bar)": df['pO2 (bar)'],
    "pN2 (bar)": df['pN2 (bar)'],
    "pCH4 (bar)": df['pCH4 (bar)'],
    "pN2O (bar)": df['pN2O (bar)'],
    "pCO (bar)": df['pCO (bar)'],
    "pO3 (bar)": df['pO3 (bar)'],
    "pSO2 (bar)": df['pSO2 (bar)'],
    "pNH3 (bar)": df['pNH3 (bar)'],
    "pC2H6 (bar)": df['pC2H6 (bar)'],
    "pNO2 (bar)": df['pNO2 (bar)'],
    "T_eff GB (K)": (F / E*SB_const)**0.25
})
X = X.replace(0, 1e-12)
T_log = LR4.predict(X)
df['C4'] = np.power(10, T_log)

df_compare = df[['T_actual (K)', 'C1', 'C2', 'C3', 'C4']]

df_compare['C1 Error (K)'] = df_compare['C1'] - df_compare['T_actual (K)']
df_compare['C2 Error (K)'] = df_compare['C2'] - df_compare['T_actual (K)']
df_compare['C3 Error (K)'] = df_compare['C3'] - df_compare['T_actual (K)']
df_compare['C4 Error (K)'] = df_compare['C4'] - df_compare['T_actual (K)']

# df_compare[['C1 Error (K)', 'C2 Error (K)', 'C3 Error (K)', 'C4 Error (K)']]
df_compare

C:\Users\henry\AppData\Local\Temp\ipykernel_1380\2102687226.py:83: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_compare['C1 Error (K)'] = df_compare['C1'] - df_compare['T_actual (K)']
C:\Users\henry\AppData\Local\Temp\ipykernel_1380\2102687226.py:84: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_compare['C2 Error (K)'] = df_compare['C2'] - df_compare['T_actual (K)']
C:\Users\henry\AppData\Local\Temp\ipykernel_1380\2102687226.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

,T_actual (K),C1,C2,C3,C4,C1 Error (K),C2 Error (K),C3 Error (K),C4 Error (K)
Mercury,440.15,449.783130,449.783130,439.396505,441.497254,9.633130,9.633130,-0.753495,1.347254
Venus,737.15,231.978600,231.978600,217.278653,219.089447,-505.171400,-505.171400,-519.871347,-518.060553
Earth,288.15,256.293787,256.293787,243.976600,248.083483,-31.856213,-31.856213,-44.173400,-40.066517
Mars,210.15,208.525725,208.525725,196.313279,197.983073,-1.624275,-1.624275,-13.836721,-12.166927


In [34]:
# s1, p1 = stats.normaltest(df['C1'])
# S2, p2 = stats.normaltest(df['C2'])
# s3, p3 = stats.normaltest(df['C3'])
# S4, p4 = stats.normaltest(df['C4'])

# if p1 <= 0.05 and p2 <= 0.05:
#     t, p_21 = stats.ttest_ind(df['C1'], df['C2'])
# else:
#     MW, p_21 = stats.mannwhitneyu(df['C1'], df['C2'])

# if p2 <= 0.05 and p3 <= 0.05:
#     t, p_32 = stats.ttest_ind(df['C2'], df['C3'])
# else:
#     MW, p_32 = stats.mannwhitneyu(df['C2'], df['C3'])

# if p3 <= 0.05 and p4 <= 0.05:
#     t, p_43 = stats.ttest_ind(df['C3'], df['C4'])
# else:
#     MW, p_43 = stats.mannwhitneyu(df['C3'], df['C4'])

# print(p_21)
# print(p_32)
# print(p_43)